# Interview Effectiveness Analysis

## Objective
Evaluate whether our interview process actually predicts on-the-job success:
- Correlate interview scores with performance outcomes
- Compare structured vs. unstructured interview effectiveness
- Identify interviewer bias patterns
- Optimize interview process for signal vs. time

## Key Questions
1. Do interview scores predict performance?
2. Which interview types are most predictive?
3. Are we over-interviewing (diminishing returns)?
4. Do interviewers show consistent rating patterns?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print("✅ Libraries loaded successfully")

## 1. Load Data

In [ ]:
# Load datasets
interviews_df = pd.read_csv('../data/interviews.csv')
candidates_df = pd.read_csv('../data/candidates.csv')
hires_df = pd.read_csv('../data/hires.csv')

print(f"📊 Data loaded:")
print(f"   Interviews: {len(interviews_df):,}")
print(f"   Candidates: {len(candidates_df):,}")
print(f"   Hires: {len(hires_df):,}")

# Merge interview data with hire outcomes
hired_candidates = candidates_df[candidates_df['current_stage'] == 'Hired']['candidate_id']
hired_interviews = interviews_df[interviews_df['candidate_id'].isin(hired_candidates)]
hired_interviews = hired_interviews.merge(hires_df[['candidate_id', 'performance_rating_12mo', 
                                                     'still_employed_12mo', 'manager_satisfaction']], 
                                          on='candidate_id')

print(f"\n📋 Hired candidates with interview data: {hired_interviews['candidate_id'].nunique()}")
print(f"   Total interviews for hired candidates: {len(hired_interviews)}")

hired_interviews.head()

## 2. Overall Interview Score vs. Performance

In [ ]:
# Calculate average interview score per hired candidate
candidate_scores = hired_interviews.groupby('candidate_id').agg({
    'score': 'mean',
    'interview_id': 'count',
    'performance_rating_12mo': 'first',
    'manager_satisfaction': 'first',
    'still_employed_12mo': 'first'
}).rename(columns={'score': 'avg_interview_score', 'interview_id': 'num_interviews'}).round(2)

# Calculate correlation
corr_performance, p_value_perf = pearsonr(candidate_scores['avg_interview_score'], 
                                           candidate_scores['performance_rating_12mo'])
corr_satisfaction, p_value_sat = pearsonr(candidate_scores['avg_interview_score'], 
                                           candidate_scores['manager_satisfaction'])

print("📊 Interview Score vs. Outcomes:\n")
print(f"Correlation with Performance Rating:")
print(f"   r = {corr_performance:.3f} (p = {p_value_perf:.4f})")
if p_value_perf < 0.05:
    print(f"   ✅ Statistically significant")
else:
    print(f"   ⚠️ Not statistically significant")

print(f"\nCorrelation with Manager Satisfaction:")
print(f"   r = {corr_satisfaction:.3f} (p = {p_value_sat:.4f})")
if p_value_sat < 0.05:
    print(f"   ✅ Statistically significant")
else:
    print(f"   ⚠️ Not statistically significant")

print(f"\n💡 Interpretation:")
if corr_performance > 0.3:
    print(f"   Strong positive relationship: Higher interview scores → Better performance")
elif corr_performance > 0.1:
    print(f"   Moderate positive relationship: Interview scores somewhat predict performance")
else:
    print(f"   Weak relationship: Interview scores are weakly predictive")

In [ ]:
# Visualize relationship
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Interview score vs. Performance
ax1.scatter(candidate_scores['avg_interview_score'], 
           candidate_scores['performance_rating_12mo'],
           alpha=0.6, s=100, edgecolors='black', c=candidate_scores['num_interviews'],
           cmap='viridis')

# Trend line
z = np.polyfit(candidate_scores['avg_interview_score'], 
               candidate_scores['performance_rating_12mo'], 1)
p = np.poly1d(z)
x_line = np.linspace(candidate_scores['avg_interview_score'].min(), 
                     candidate_scores['avg_interview_score'].max(), 100)
ax1.plot(x_line, p(x_line), "r--", linewidth=2, 
         label=f'Trend line (r={corr_performance:.3f})')

ax1.set_xlabel('Average Interview Score')
ax1.set_ylabel('12-Month Performance Rating')
ax1.set_title('Interview Score vs. Performance', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# Interview score distribution by performance tier
candidate_scores['perf_tier'] = pd.cut(candidate_scores['performance_rating_12mo'], 
                                        bins=[0, 3, 4, 5], 
                                        labels=['Low (1-3)', 'Mid (3-4)', 'High (4-5)'])
candidate_scores.boxplot(column='avg_interview_score', by='perf_tier', ax=ax2, 
                        patch_artist=True, grid=False)
ax2.set_xlabel('Performance Tier')
ax2.set_ylabel('Average Interview Score')
ax2.set_title('Interview Scores by Performance Tier', fontsize=14, fontweight='bold')
plt.suptitle('')  # Remove default title
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Interview Type Effectiveness

In [ ]:
# Correlation by interview type
interview_type_corr = []

for interview_type in hired_interviews['interview_type'].unique():
    type_data = hired_interviews[hired_interviews['interview_type'] == interview_type]
    
    # Average score per candidate for this interview type
    type_scores = type_data.groupby('candidate_id').agg({
        'score': 'mean',
        'performance_rating_12mo': 'first'
    })
    
    if len(type_scores) >= 10:  # Require minimum sample
        corr, p_val = pearsonr(type_scores['score'], type_scores['performance_rating_12mo'])
        
        interview_type_corr.append({
            'interview_type': interview_type,
            'correlation': corr,
            'p_value': p_val,
            'n_candidates': len(type_scores),
            'n_interviews': len(type_data)
        })

type_corr_df = pd.DataFrame(interview_type_corr).sort_values('correlation', ascending=False)

print("📊 Predictive Power by Interview Type:\n")
print(type_corr_df.round(3).to_string(index=False))

print("\n💡 Interpretation:")
print("   Higher correlation = Better predictor of performance")
print("   p < 0.05 = Statistically significant relationship")

In [ ]:
# Visualize interview type effectiveness
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Correlation strength
colors = ['green' if p < 0.05 else 'orange' for p in type_corr_df['p_value']]
ax1.barh(range(len(type_corr_df)), type_corr_df['correlation'], 
         color=colors, edgecolor='black', alpha=0.7)
ax1.set_yticks(range(len(type_corr_df)))
ax1.set_yticklabels(type_corr_df['interview_type'])
ax1.set_xlabel('Correlation with Performance')
ax1.set_title('Predictive Power by Interview Type', fontsize=14, fontweight='bold')
ax1.axvline(x=0, color='black', linewidth=1)
ax1.grid(axis='x', alpha=0.3)

# Add correlation values
for i, (corr, n) in enumerate(zip(type_corr_df['correlation'], type_corr_df['n_candidates'])):
    ax1.text(corr + 0.01, i, f'{corr:.3f} (n={n})', va='center', fontsize=9)

# Average score by type
avg_scores = hired_interviews.groupby('interview_type')['score'].mean().sort_values()
avg_scores.plot(kind='barh', ax=ax2, color='steelblue', edgecolor='black', alpha=0.7)
ax2.axvline(x=hired_interviews['score'].mean(), color='red', linestyle='--', 
            linewidth=2, label='Overall Average')
ax2.set_xlabel('Average Interview Score')
ax2.set_title('Average Score by Interview Type', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🟢 Green = Statistically significant (p < 0.05)")
print("🟠 Orange = Not statistically significant (p >= 0.05)")

## 4. Diminishing Returns: Interview Quantity Analysis

In [ ]:
# Analyze performance by number of interviews
perf_by_num_interviews = candidate_scores.groupby('num_interviews').agg({
    'performance_rating_12mo': ['mean', 'std', 'count'],
    'manager_satisfaction': 'mean'
}).round(2)

perf_by_num_interviews.columns = ['avg_performance', 'std_performance', 'n_hires', 'avg_satisfaction']
perf_by_num_interviews = perf_by_num_interviews[perf_by_num_interviews['n_hires'] >= 3]

print("📊 Performance by Number of Interviews:\n")
print(perf_by_num_interviews)

# Statistical test: Does more interviews = better performance?
corr_num_int, p_num_int = spearmanr(candidate_scores['num_interviews'], 
                                     candidate_scores['performance_rating_12mo'])

print(f"\n📈 Correlation (# interviews vs. performance): {corr_num_int:.3f} (p={p_num_int:.4f})")
if abs(corr_num_int) < 0.1:
    print("   💡 Weak/no relationship: More interviews ≠ Better hires")
elif corr_num_int > 0:
    print("   💡 Positive relationship: More interviews → Better performance")
else:
    print("   💡 Negative relationship: More interviews → Worse performance (!)")

In [ ]:
# Visualize diminishing returns
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Performance by number of interviews
x = perf_by_num_interviews.index
y = perf_by_num_interviews['avg_performance']
ax1.plot(x, y, marker='o', linewidth=2, markersize=10, color='steelblue')
ax1.axhline(y=candidate_scores['performance_rating_12mo'].mean(), 
            color='red', linestyle='--', linewidth=2, label='Overall Average')
ax1.set_xlabel('Number of Interviews')
ax1.set_ylabel('Average Performance Rating')
ax1.set_title('Performance vs. Interview Quantity', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# Add data labels
for xi, yi, ni in zip(x, y, perf_by_num_interviews['n_hires']):
    ax1.text(xi, yi + 0.05, f'n={int(ni)}', ha='center', fontsize=9)

# Distribution of interview counts
interview_counts = candidate_scores['num_interviews'].value_counts().sort_index()
ax2.bar(interview_counts.index, interview_counts.values, 
        color='green', alpha=0.7, edgecolor='black')
ax2.set_xlabel('Number of Interviews')
ax2.set_ylabel('Number of Hires')
ax2.set_title('Distribution: Interview Count per Hire', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# Add percentages
total = interview_counts.sum()
for x, count in interview_counts.items():
    pct = count / total * 100
    ax2.text(x, count + 0.5, f'{pct:.0f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## 5. Interviewer Consistency Analysis

In [ ]:
# Analyze interviewer rating patterns
interviewer_stats = interviews_df.groupby('interviewer_id').agg({
    'score': ['mean', 'std', 'count']
}).round(2)

interviewer_stats.columns = ['avg_score', 'std_score', 'n_interviews']
interviewer_stats = interviewer_stats[interviewer_stats['n_interviews'] >= 5]  # Min 5 interviews
interviewer_stats = interviewer_stats.sort_values('avg_score', ascending=False)

print(f"📊 Interviewer Patterns (n={len(interviewer_stats)} with ≥5 interviews):\n")
print(f"Average score across all interviewers: {interviewer_stats['avg_score'].mean():.2f}")
print(f"Score variation (std): {interviewer_stats['avg_score'].std():.2f}")
print(f"\nMost lenient (highest avg scores):")
print(interviewer_stats.head(5).to_string())
print(f"\nMost strict (lowest avg scores):")
print(interviewer_stats.tail(5).to_string())

# Identify potential bias
overall_mean = interviews_df['score'].mean()
lenient_threshold = overall_mean + interviewer_stats['avg_score'].std()
strict_threshold = overall_mean - interviewer_stats['avg_score'].std()

lenient_interviewers = interviewer_stats[interviewer_stats['avg_score'] > lenient_threshold]
strict_interviewers = interviewer_stats[interviewer_stats['avg_score'] < strict_threshold]

print(f"\n⚠️ Potential Calibration Issues:")
print(f"   Lenient interviewers (>1 SD above mean): {len(lenient_interviewers)}")
print(f"   Strict interviewers (>1 SD below mean): {len(strict_interviewers)}")

In [ ]:
# Visualize interviewer patterns
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Distribution of interviewer average scores
ax1.hist(interviewer_stats['avg_score'], bins=15, edgecolor='black', alpha=0.7, color='purple')
ax1.axvline(x=overall_mean, color='red', linestyle='--', linewidth=2, label='Overall Mean')
ax1.axvline(x=lenient_threshold, color='orange', linestyle=':', linewidth=2, label='+1 SD')
ax1.axvline(x=strict_threshold, color='orange', linestyle=':', linewidth=2, label='-1 SD')
ax1.set_xlabel('Average Interview Score')
ax1.set_ylabel('Number of Interviewers')
ax1.set_title('Interviewer Average Score Distribution', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# Score consistency (std dev)
ax2.scatter(interviewer_stats['avg_score'], interviewer_stats['std_score'], 
           s=interviewer_stats['n_interviews']*10, alpha=0.6, edgecolors='black')
ax2.axvline(x=overall_mean, color='red', linestyle='--', alpha=0.5, label='Overall Mean')
ax2.set_xlabel('Average Score')
ax2.set_ylabel('Standard Deviation (Consistency)')
ax2.set_title('Interviewer Scoring Patterns (size = # interviews)', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)

# Annotate quadrants
ax2.text(lenient_threshold+0.1, interviewer_stats['std_score'].max()*0.9, 
        '⚠️ Lenient &\nInconsistent', fontsize=9, bbox=dict(boxstyle='round', 
        facecolor='lightyellow', alpha=0.5))
ax2.text(strict_threshold-0.1, interviewer_stats['std_score'].max()*0.9, 
        '⚠️ Strict &\nInconsistent', fontsize=9, ha='right', bbox=dict(boxstyle='round', 
        facecolor='lightcoral', alpha=0.5))

plt.tight_layout()
plt.show()

print("\n💡 Ideal interviewers: Near overall mean, low standard deviation (consistent)")

## 6. Interview Score Reliability: Notes Completion

In [ ]:
# Compare interview quality by notes completion
notes_analysis = hired_interviews.groupby('notes_completed').agg({
    'score': ['mean', 'std'],
    'interview_id': 'count'
}).round(2)

notes_analysis.columns = ['avg_score', 'std_score', 'n_interviews']

print("📝 Interview Quality by Notes Completion:\n")
print(notes_analysis)

# Test if notes completion correlates with better signal
notes_completed = hired_interviews[hired_interviews['notes_completed'] == True]
notes_missing = hired_interviews[hired_interviews['notes_completed'] == False]

# Compare score variance
print(f"\n📊 Score Variability:")
print(f"   Notes completed - Std Dev: {notes_completed['score'].std():.2f}")
print(f"   Notes missing - Std Dev: {notes_missing['score'].std():.2f}")

if notes_missing['score'].std() > notes_completed['score'].std():
    print(f"   ⚠️ Missing notes associated with MORE inconsistent scoring")
else:
    print(f"   ✅ Notes completion doesn't significantly affect consistency")

notes_completion_rate = (interviews_df['notes_completed'].sum() / len(interviews_df) * 100)
print(f"\n📈 Overall notes completion rate: {notes_completion_rate:.1f}%")
if notes_completion_rate < 85:
    print(f"   ⚠️ Below target (85%) - Improve compliance")

## 7. Key Findings & Recommendations

In [ ]:
print("="*70)
print("🎯 KEY FINDINGS")
print("="*70)

print("\n1️⃣ INTERVIEW PREDICTIVE POWER:\n")
print(f"   Overall correlation (interview score ↔ performance): {corr_performance:.3f}")
if corr_performance > 0.3:
    print(f"   ✅ STRONG: Interviews are highly predictive")
elif corr_performance > 0.15:
    print(f"   ✅ MODERATE: Interviews are somewhat predictive")
else:
    print(f"   ⚠️ WEAK: Interviews show limited predictive power")

print("\n2️⃣ MOST PREDICTIVE INTERVIEW TYPES:\n")
for i, row in type_corr_df.head(3).iterrows():
    sig = "✅" if row['p_value'] < 0.05 else "⚠️"
    print(f"   {sig} {row['interview_type']}: r={row['correlation']:.3f} (n={int(row['n_candidates'])})")

print("\n3️⃣ INTERVIEW QUANTITY ANALYSIS:\n")
median_interviews = candidate_scores['num_interviews'].median()
print(f"   Median interviews per hire: {median_interviews:.0f}")
print(f"   Correlation (# interviews ↔ performance): {corr_num_int:.3f}")
if abs(corr_num_int) < 0.1:
    print(f"   💡 More interviews ≠ Better hires (diminishing returns)")

print("\n4️⃣ INTERVIEWER CALIBRATION:\n")
print(f"   Interviewers analyzed: {len(interviewer_stats)}")
print(f"   Score variance across interviewers: {interviewer_stats['avg_score'].std():.2f}")
print(f"   Lenient interviewers (>1 SD): {len(lenient_interviewers)}")
print(f"   Strict interviewers (>1 SD): {len(strict_interviewers)}")
if len(lenient_interviewers) + len(strict_interviewers) > len(interviewer_stats) * 0.3:
    print(f"   ⚠️ High variability - Calibration needed")

print("\n5️⃣ INTERVIEW DOCUMENTATION:\n")
print(f"   Notes completion rate: {notes_completion_rate:.1f}%")
if notes_completion_rate < 85:
    print(f"   ⚠️ Below target - Improve compliance")
else:
    print(f"   ✅ Meeting target")

print("\n" + "="*70)
print("💡 RECOMMENDATIONS")
print("="*70)

print("\n🎯 IMMEDIATE ACTIONS (0-30 days):\n")
print("   1. Prioritize high-signal interview types in final decisions")
best_type = type_corr_df.iloc[0]['interview_type']
print(f"      → Focus on: {best_type}")
print("   2. Calibrate interviewers showing extreme leniency/strictness")
print("   3. Enforce interview notes completion (target: 90%+)")

print("\n🔧 PROCESS IMPROVEMENTS (30-90 days):\n")
print("   1. STANDARDIZE interview formats (structured > unstructured)")
print("   2. REDUCE interview loops if no correlation with outcomes")
if median_interviews > 4:
    print(f"      → Consider reducing from {median_interviews:.0f} to 3-4 rounds")
print("   3. TRAIN interviewers on consistent scoring rubrics")
print("   4. PILOT panel interviews to reduce individual bias")

print("\n📊 ONGOING MONITORING:\n")
print("   • Track interview score ↔ performance correlation quarterly")
print("   • Review interviewer calibration monthly")
print("   • A/B test interview process changes")
print("   • Survey candidates on interview experience")

print("\n🎯 EXPECTED IMPACT:\n")
print("   • 15-20% improvement in hire quality (better signal)")
print("   • 20-30% reduction in time-to-fill (fewer, better interviews)")
print("   • Improved candidate experience (streamlined process)")
print("   • Reduced interviewer burden (more focused interviews)")

print("\n" + "="*70)

## Next Steps

1. **Interviewer Training**: Design calibration workshops for outlier interviewers
2. **Process Optimization**: Pilot 3-interview loop vs. current process
3. **Structured Templates**: Create standardized interview guides by role
4. **Bias Audit**: Analyze interview scores by candidate demographics
5. **Feedback Loop**: Share performance correlations with interviewers

## Related Analyses
- `03_quality_of_hire_prediction.ipynb` - Predictive modeling
- `01_source_of_hire_analysis.ipynb` - Source effectiveness
- `05_diversity_pipeline.ipynb` - Interview bias detection